# R6: TD-MPC2 encoder lens (Colab runner)

Runs, in order, the committed scripts for R6 under `prereg/r6.md` (tag `prereg-r6`)
and `prereg/r6-deviations.md` (tag `prereg-r6-d1`):

1. `extract.py list` / `extract` / `lens`: checkpoints, first-layer weights, lens.
2. `collect.py collect`: policy-only episodes in DMControl (D4), returns vs published.
3. `collect.py consistency`: position-0 latent-consistency test (D3).

This notebook only calls the scripts; it computes nothing itself. It does **not**
compute the R6 criterion or gate G1. Both scripts refuse to run unless both annotated
tags exist and the pre-registration files match them.

Runtime: CPU is enough. `collect` is the slow step (5 tasks x 3 seeds x 50 episodes x 500 steps).

In [ ]:
# 1. Clone the repository with its tags. If the repository is private, add a GitHub
#    token as the Colab secret GITHUB_TOKEN (key icon, left sidebar).
import os, subprocess
BRANCH = "claude/loving-johnson-wk9yb3"
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None
url = (f"https://{token}@github.com/binoygeorge97/layernorm-lens" if token
       else "https://github.com/binoygeorge97/layernorm-lens")
if not os.path.isdir("layernorm-lens"):
    subprocess.run(["git", "clone", "--branch", BRANCH, url, "layernorm-lens"], check=True)
%cd layernorm-lens
!git fetch --tags -q origin
!git log --oneline -1
!for t in prereg-r6 prereg-r6-d1; do printf "%-14s " $t; git cat-file -t $t && git rev-parse $t^{commit}; done

In [ ]:
# 2. Pinned environment: core (JAX-only) + R6 extras (CPU torch, dm_control, mujoco).
!pip install -q torch==2.14.0 --index-url https://download.pytorch.org/whl/cpu
!pip install -q -r requirements.txt -r requirements-r6.txt
# If pip reports that preinstalled Colab packages were replaced, restart the runtime
# (Runtime > Restart session), then re-run cell 1 (it will skip the clone) and continue.

In [ ]:
# 3. Versions and headless MuJoCo.
import os
os.environ["MUJOCO_GL"] = "egl"
!python -c "import jax, numpy, torch, mujoco, dm_control; print('jax', jax.__version__, 'numpy', numpy.__version__, 'torch', torch.__version__, 'mujoco', mujoco.__version__)"

In [ ]:
# 4. TD-MPC2 source at the pinned commit (published learning curves for the returns check).
!test -d checkpoints/tdmpc2_src || git clone -q https://github.com/nicklashansen/tdmpc2 checkpoints/tdmpc2_src
!git -C checkpoints/tdmpc2_src checkout -q e9f59321933cbc8e11a002b842adc7d4ffae8ff1 && git -C checkpoints/tdmpc2_src rev-parse HEAD

In [ ]:
# 5. Pipeline checks (not outcome metrics). Versions differ from the golden files here,
#    so the regression test runs in tolerance mode.
!python -m pytest -q tests/test_lens.py
!LENS_TOL=1 python -m pytest -q tests/test_core_regression.py

In [ ]:
# 6. Checkpoint listing (downloads nothing). Check it before continuing.
!python experiments/r6_tdmpc2/extract.py --config experiments/r6_tdmpc2/config.yaml list

In [ ]:
# 7. Download the configured seeds, check the released layout, save the first layer.
!python experiments/r6_tdmpc2/extract.py --config experiments/r6_tdmpc2/config.yaml extract

In [ ]:
# 8. Lens of each first layer (z*, principal widths, kappa, degenerate).
!python experiments/r6_tdmpc2/extract.py --config experiments/r6_tdmpc2/config.yaml lens
import pandas as pd
pd.read_csv("results/r6/lens_summary.csv")

In [ ]:
# 9. Policy-only data collection (D4). Long-running.
!python experiments/r6_tdmpc2/collect.py --config experiments/r6_tdmpc2/config.yaml collect
pd.read_csv("results/r6/returns.csv")

In [ ]:
# 10. Position-0 latent-consistency test (D3). Exit status 2 and "STOP" mean identity
#     was rejected for some task: do not continue; consult the author.
!python experiments/r6_tdmpc2/collect.py --config experiments/r6_tdmpc2/config.yaml consistency; echo "exit status: $?"
pd.read_csv("results/r6/consistency.csv")

In [ ]:
# 11. Save outputs. Small summaries (to commit): results/r6/*.csv and meta_*.json.
#     Large files (data/r6, results/r6/weights, results/r6/lens) stay out of git.
!zip -q -r r6_summaries.zip results/r6/*.csv results/r6/meta_*.json results/r6/consistency.json
!zip -q -r r6_full.zip results/r6 data/r6
from google.colab import files
files.download("r6_summaries.zip")